In [1]:
import pandas as pd

euro_matches = pd.read_pickle('fast_actions_euro.pkl')
fifa_matches = pd.read_pickle('fast_actions_fifa.pkl')
len(euro_matches)

2917

In [2]:
len(fifa_matches)

3694

In [3]:
def set_good_end_values(df):
    opponent_penalty_area = ['K3', 'K4', 'K5', 'K6', 'L3', 'L4', 'L5', 'L6']
    newDf = df
    newDf['good_end'] = [1 if (end_area in opponent_penalty_area) or (shots > 0) else 0 for end_area, shots in zip(df['end_area'].values, df['num_of_shot'].values)]
    return newDf
    

In [4]:
euro_matches = set_good_end_values(euro_matches)
fifa_matches = set_good_end_values(fifa_matches)
len(euro_matches[euro_matches['good_end'] == 1])

345

In [5]:
len(fifa_matches[fifa_matches['good_end'] == 1])

418

In [14]:
len(euro_matches[euro_matches['num_of_shot'] > 0])

101

In [15]:
len(fifa_matches[fifa_matches['num_of_shot'] > 0])

104

In [6]:
from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler

def get_sets_for_model(df_train, df_test):
    df_train = shuffle(df_train)
    df_test = shuffle(df_test)

    # scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))

    df_train_ids = df_train['id'].values  # type: ignore
    df_test_ids = df_test['id'].values  # type: ignore
    df_train_teams = df_train['team'].values  # type: ignore
    df_test_teams = df_test['team'].values  # type: ignore
    df_train_good_ends = df_train['good_end'].values  # type: ignore
    df_test_good_ends = df_test['good_end'].values  # type: ignore

    df_train = df_train.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'good_end']) # type: ignore
    df_test = df_test.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'good_end']) # type: ignore

    df_train_columns = df_train.columns.sort_values()
    df_test_columns = df_test.columns.sort_values()

    # df_train = scaler.fit_transform(df_train)
    # df_test = scaler.fit_transform(df_test)

    df_train = pd.DataFrame(df_train, columns=df_train_columns)
    df_test = pd.DataFrame(df_test, columns=df_test_columns)

    df_train['id'] = df_train_ids
    df_test['id'] = df_test_ids
    df_train['team'] = df_train_teams
    df_test['team'] = df_test_teams
    df_train['good_end'] = df_train_good_ends
    df_test['good_end'] = df_test_good_ends

    y_train = df_train['good_end']
    X_train = df_train.drop(columns=['good_end'])

    y_test = df_test['good_end']
    X_test = df_test.drop(columns=['good_end'])

    return X_train, X_test, y_train, y_test

In [7]:
X_train, X_test, y_train, y_test = get_sets_for_model(fifa_matches, euro_matches)

In [8]:
from xgboost import XGBClassifier

def train_model(X_train, X_test, y_train, y_test):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot'])
    model.fit(input_X_train, y_train)

    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot'])
    XGB_predictions = model.predict(input_X_test)

    return model, XGB_predictions, input_X_test

model, pred, input_test = train_model(X_train, X_test, y_train, y_test)

In [9]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_model(y_test, pred):
    print('Confusion matrix')
    print(confusion_matrix(y_test, pred))
    print('Classification report')
    print(classification_report(y_test, pred))
    print('Accuracy= ', accuracy_score(y_test, pred))

evaluate_model(y_test, pred)

Confusion matrix
[[2499   73]
 [ 306   39]]
Classification report
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      2572
           1       0.35      0.11      0.17       345

    accuracy                           0.87      2917
   macro avg       0.62      0.54      0.55      2917
weighted avg       0.83      0.87      0.84      2917

Accuracy=  0.8700719917723688


In [10]:
def feature_importances_raport(input_X_test, model, set_title):
    result = pd.DataFrame(columns=['name', 'value'])
    result['name'] = input_X_test.columns
    result['value'] = model.feature_importances_

    result = result.sort_values(by=['value'], ascending=False)

    result.to_csv(f'MBV_feature_importances_{set_title}.csv')
    print(result)
    return result

feature_importances = feature_importances_raport(input_test, model, 'fast_sequences_test3')

                    name     value
10         num_of_events  0.130808
4           num_of_carry  0.106895
13           num_of_pass  0.094407
1   num_of_ball_receipt*  0.077141
12     num_of_miscontrol  0.075718
8            num_of_duel  0.075691
7         num_of_dribble  0.072634
11   num_of_interception  0.072260
0               duration  0.071556
6    num_of_dispossessed  0.070279
2   num_of_ball_recovery  0.060549
5       num_of_clearance  0.048374
3           num_of_block  0.043689
9           num_of_error  0.000000


In [11]:
def generate_scores(model, X_test, X_train, y_train, set_title):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot'])
    model.fit(input_X_train, y_train)
    
    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot'])
    model = model.predict_proba(input_X_test)

    X_test['score_positive'] = [m[1] for m in model]
    X_test['score_negative'] = [m[0] for m in model]

    X_test.to_csv(f'{set_title}_evaluated.csv')

    return X_test

df_with_scores = generate_scores(model, X_test, X_train, y_train, 'fast_sequences_test3')

In [13]:
df_with_scores[df_with_scores['num_of_shot'] > 0]

,duration,num_of_ball_receipt*,num_of_ball_recovery,num_of_block,num_of_carry,num_of_clearance,num_of_dispossessed,num_of_dribble,num_of_duel,num_of_error,num_of_events,num_of_interception,num_of_miscontrol,num_of_pass,num_of_shot,id,team,score_positive,score_negative
0,14,3,0,0,2,0,0,0,0,0,9,0,0,3,1,bb294dab-1894-4d45-874a-e5b32bcdf89a-2ad830f5-...,England,0.289707,0.710293
0,125,2,1,0,4,0,0,1,0,0,11,0,0,2,1,e95093dc-1f5a-4268-bdd5-0ecb100f6952-b6976187-...,Croatia,0.972163,0.027837
0,40,2,1,0,3,0,0,0,0,0,9,0,0,2,1,47f0d7f9-c8d9-42c9-aef1-a14e0b59cffb-1ff8f1f1-...,Italy,0.473016,0.526984
0,115,3,0,0,4,0,0,1,0,0,12,0,0,3,1,6c3f47c2-5dd5-4a46-bb9c-cef7253588d2-ceb030c3-...,Denmark,0.126063,0.873937
0,7,1,1,0,2,0,0,0,0,0,6,0,0,1,1,eea38b4e-9d3e-447c-8e0d-ef05932198b2-35f6e898-...,Denmark,0.297724,0.702276
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,93,2,1,0,3,0,0,0,0,0,9,0,0,2,1,191ce5a4-7792-4236-8d2c-cebcb0737a68-2c29d8d8-...,Switzerland,0.705710,0.294290
0,48,1,1,0,2,0,0,0,0,0,6,0,0,1,1,af23819e-c6f0-42b1-aeb4-d533f5f96cff-27804b3d-...,Belgium,0.705988,0.294012
0,16,3,0,0,4,0,0,1,0,0,12,0,0,3,1,632ed167-22b3-4f4b-b0b3-56d061f5d8c9-1e90a015-...,Portugal,0.088490,0.911510
0,77,4,0,0,3,0,0,0,0,0,12,0,0,4,1,a0943f19-47e0-48db-b8e7-36e7ee3a94c5-e3a46723-...,Italy,0.603592,0.396408
